# 07 – Longitudinal risk prediction and trend (LSTM)

**Data:** Framingham longitudinal teaching dataset (riskCommunicator): 4,434 participants, up to 3 exams ~6 years apart, 24-year follow-up.
**Note:** teaching version (anonymised); used to build and demonstrate the longitudinal layer.

**Goals:**
1. Does a patient's exam history improve 10-year CVD prediction over a single snapshot?
2. Classify each patient's risk trend (increasing / stable / decreasing) and test whether it is clinically meaningful

## Step 1 – Load data

In [7]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # allow PyTorch + XGBoost together on Mac
os.environ["OMP_NUM_THREADS"] = "1"

import warnings; warnings.filterwarnings("ignore")
from xgboost import XGBClassifier             # import XGBoost BEFORE torch
import numpy as np, pandas as pd, pyreadr, torch, torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix

torch.manual_seed(42); np.random.seed(42)

df = pyreadr.read_r("../data/framingham.rda")["framingham"]
print(df.shape)
print(df.groupby("RANDID")["PERIOD"].nunique().value_counts().sort_index())

(11627, 39)
PERIOD
1     447
2     781
3    3206
Name: count, dtype: int64


## Step 2 – Build the cohort

- Patients with **all 3 exams**
- Exclude anyone who already had CVD, CHD or stroke by exam 3
- **Outcome:** CVD within 10 years after exam 3

In [8]:
TEN_YEARS = 3652
FEATS = ["SEX", "AGE", "SYSBP", "DIABP", "TOTCHOL", "CURSMOKE", "CIGPDAY",
         "BMI", "DIABETES", "BPMEDS", "HEARTRTE", "GLUCOSE"]

n_exams = df.groupby("RANDID")["PERIOD"].nunique()
d = df[df["RANDID"].isin(n_exams[n_exams == 3].index)].sort_values(["RANDID", "PERIOD"])

exam3 = d[d["PERIOD"] == 3].set_index("RANDID")
healthy_at_exam3 = (~((exam3["CVD"] == 1) & (exam3["TIMECVD"] <= exam3["TIME"]))
                    & (exam3["PREVCHD"] == 0) & (exam3["PREVSTRK"] == 0))
exam3 = exam3[healthy_at_exam3]
ids = exam3.index
y = ((exam3["CVD"] == 1) & (exam3["TIMECVD"] <= exam3["TIME"] + TEN_YEARS)).astype(int)
d = d[d["RANDID"].isin(ids)].copy()

print("Patients:", len(ids), "| 10-year CVD rate:", round(y.mean(), 3))

Patients: 2806 | 10-year CVD rate: 0.139


## Step 3 – Split patients and build sequences

- Split by **patient** (60 / 20 / 20), so no patient appears in two sets
- Each patient = a sequence of 3 exams × 13 inputs (12 features + years since previous exam)
- Missing values and scaling learned from training patients only

In [9]:
train_ids, tmp_ids = train_test_split(ids, test_size=0.4, stratify=y.loc[ids], random_state=42)
cal_ids, test_ids = train_test_split(tmp_ids, test_size=0.5, stratify=y.loc[tmp_ids], random_state=42)

train_rows = d[d["RANDID"].isin(train_ids)]
imp = SimpleImputer(strategy="median").fit(train_rows[FEATS])
sc = StandardScaler().fit(imp.transform(train_rows[FEATS]))
d[FEATS] = sc.transform(imp.transform(d[FEATS]))
d["GAP"] = (d.groupby("RANDID")["TIME"].diff().fillna(0) / 365.25 - 3) / 3
SEQ_FEATS = FEATS + ["GAP"]

by_patient = {pid: g[SEQ_FEATS].values for pid, g in d.groupby("RANDID")}
def to_seq(id_list):
    return torch.tensor(np.stack([by_patient[i] for i in id_list]), dtype=torch.float32)

X_train, X_cal, X_test = to_seq(train_ids), to_seq(cal_ids), to_seq(test_ids)
y_train, y_cal, y_test = [torch.tensor(y.loc[i].values, dtype=torch.float32)
                          for i in (train_ids, cal_ids, test_ids)]
print(X_train.shape, X_cal.shape, X_test.shape)

torch.Size([1683, 3, 13]) torch.Size([561, 3, 13]) torch.Size([562, 3, 13])


## Step 4 – Baselines (snapshot vs history)

In [10]:
last_exam = lambda X: X[:, 2, :len(FEATS)].numpy()
all_exams = lambda X: X.reshape(len(X), -1).numpy()

lr_snap = LogisticRegression(max_iter=2000).fit(last_exam(X_train), y_train)
xgb_snap = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03, subsample=0.8,
                         random_state=42, n_jobs=1).fit(last_exam(X_train), y_train)
lr_hist = LogisticRegression(max_iter=2000).fit(all_exams(X_train), y_train)

results = {
    "LR – last exam only":      roc_auc_score(y_test, lr_snap.predict_proba(last_exam(X_test))[:, 1]),
    "XGBoost – last exam only": roc_auc_score(y_test, xgb_snap.predict_proba(last_exam(X_test))[:, 1]),
    "LR – all 3 exams (flat)":  roc_auc_score(y_test, lr_hist.predict_proba(all_exams(X_test))[:, 1]),
}
{k: round(v, 3) for k, v in results.items()}

{'LR – last exam only': 0.735,
 'XGBoost – last exam only': 0.679,
 'LR – all 3 exams (flat)': 0.752}

## Step 5 – LSTM on the 3-exam sequence

The LSTM reads the exams in order and learns how changes over time relate to future CVD. Best epoch chosen on the calibration set.

In [11]:
class RiskLSTM(nn.Module):
    def __init__(self, in_dim, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, batch_first=True)
        self.drop = nn.Dropout(0.3)
        self.out = nn.Linear(hidden, 1)
    def forward(self, x):
        h, _ = self.lstm(x)
        return self.out(self.drop(h[:, -1])).squeeze(-1)

torch.manual_seed(42)
lstm = RiskLSTM(len(SEQ_FEATS))
opt = torch.optim.Adam(lstm.parameters(), lr=0.005, weight_decay=1e-4)
best_auc, best_state = 0, None
for epoch in range(200):
    lstm.train(); opt.zero_grad()
    loss = nn.functional.binary_cross_entropy_with_logits(lstm(X_train), y_train)
    loss.backward(); opt.step()
    lstm.eval()
    with torch.no_grad():
        cal_auc = roc_auc_score(y_cal, lstm(X_cal))
    if cal_auc > best_auc:
        best_auc, best_state = cal_auc, {k: v.clone() for k, v in lstm.state_dict().items()}

lstm.load_state_dict(best_state); lstm.eval()
with torch.no_grad():
    p_lstm = torch.sigmoid(lstm(X_test)).numpy()
results["LSTM – 3-exam sequence"] = roc_auc_score(y_test, p_lstm)
pd.Series(results).round(3)

LR – last exam only         0.735
XGBoost – last exam only    0.679
LR – all 3 exams (flat)     0.752
LSTM – 3-exam sequence      0.761
dtype: float64

## Step 6 – Is the LSTM gain real? (bootstrap 95% CI)

In [12]:
p_snap = lr_snap.predict_proba(last_exam(X_test))[:, 1]
yt = y_test.numpy(); rng = np.random.default_rng(0); diffs = []
for _ in range(1000):
    i = rng.integers(0, len(yt), len(yt))
    if len(np.unique(yt[i])) < 2: continue
    diffs.append(roc_auc_score(yt[i], p_lstm[i]) - roc_auc_score(yt[i], p_snap[i]))
print("AUC gain:", round(np.mean(diffs), 3), "| 95% CI:", np.percentile(diffs, [2.5, 97.5]).round(3))

AUC gain: 0.025 | 95% CI: [-0.004  0.059]


## Step 7 – Risk trend per patient

- Compute each patient's risk at every exam, **holding age fixed at exam 1**, so the trend reflects **modifiable factors** (BP, cholesterol, smoking, BMI…) rather than just getting older
- Trend = change from exam 1 to exam 3: **> +3 pts increasing, < −3 pts decreasing, otherwise stable**
- Check: do trend groups differ in actual 10-year CVD?

In [13]:
risk_inputs = d[FEATS].copy()
risk_inputs["AGE"] = d.groupby("RANDID")["AGE"].transform("first")
d["mod_risk"] = lr_snap.predict_proba(risk_inputs.values)[:, 1] * 100

R = d.pivot(index="RANDID", columns="PERIOD", values="mod_risk")
change = R[3] - R[1]
trend = pd.cut(change, [-np.inf, -3, 3, np.inf], labels=["decreasing", "stable", "increasing"])

summary = pd.DataFrame({"trend": trend, "cvd_10y": y.loc[R.index]})
summary.groupby("trend", observed=True)["cvd_10y"].agg(patients="size", cvd_rate="mean").round(3)

,patients,cvd_rate
trend,,
decreasing,218,0.216
stable,2040,0.098
increasing,548,0.259


## Step 8 – Can we forecast the next trend?

Using only exams 1–2, predict whether the exam 2 → 3 change will be decreasing / stable / increasing.
76% of patients are "stable", so we use **class weights** and report **balanced accuracy** (chance = 33%).

In [14]:
TH = 3
lab = pd.cut(R[3] - R[2], [-np.inf, -TH, TH, np.inf], labels=[0, 1, 2]).astype(int)

def first_two(id_list):
    return torch.tensor(np.stack([np.c_[by_patient[i][:2], R.loc[i, [1, 2]].values.reshape(-1, 1)]
                                  for i in id_list]), dtype=torch.float32)

T_train, T_cal, T_test = first_two(train_ids), first_two(cal_ids), first_two(test_ids)
L_train, L_cal, L_test = [torch.tensor(lab.loc[i].values) for i in (train_ids, cal_ids, test_ids)]

class TrendLSTM(nn.Module):
    def __init__(self, in_dim, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, batch_first=True)
        self.drop = nn.Dropout(0.3); self.out = nn.Linear(hidden, 3)
    def forward(self, x):
        h, _ = self.lstm(x); return self.out(self.drop(h[:, -1]))

torch.manual_seed(42)
tl = TrendLSTM(T_train.shape[2])
weights = torch.tensor(len(L_train) / (3 * np.bincount(L_train.numpy())), dtype=torch.float32)
opt = torch.optim.Adam(tl.parameters(), lr=0.005, weight_decay=1e-4)
best, best_state = 0, None
for epoch in range(300):
    tl.train(); opt.zero_grad()
    nn.functional.cross_entropy(tl(T_train), L_train, weight=weights).backward(); opt.step()
    tl.eval()
    with torch.no_grad():
        b = balanced_accuracy_score(L_cal, tl(T_cal).argmax(1))
    if b > best: best, best_state = b, {k: v.clone() for k, v in tl.state_dict().items()}

tl.load_state_dict(best_state); tl.eval()
with torch.no_grad(): pred = tl(T_test).argmax(1)
print("Accuracy:", round(accuracy_score(L_test, pred), 3),
      "| Balanced accuracy:", round(balanced_accuracy_score(L_test, pred), 3),
      "| Always-'stable' accuracy:", round((L_test == 1).float().mean().item(), 3))
print(confusion_matrix(L_test, pred))

Accuracy: 0.667 | Balanced accuracy: 0.64 | Always-'stable' accuracy: 0.762
[[ 40   4   5]
 [ 50 301  77]
 [ 36  15  34]]


## Summary

| Model | Test AUC (10-year CVD) |
|---|---|
| XGBoost – last exam | 0.679 |
| Logistic Regression – last exam | 0.735 |
| Logistic Regression – all 3 exams | 0.752 |
| **LSTM – 3-exam sequence** | **0.761** |

- **History helps:** LSTM +0.025 AUC over a single snapshot (95% CI −0.004 to +0.059) – promising but not statistically conclusive
- **Trend is clinically meaningful:** 10-year CVD 9.8% (stable) vs 25.9% (increasing) vs 21.6% (decreasing)
- **Trend forecasting is hard:** balanced accuracy 64% (chance 33%); plain accuracy (67%) is below the always-"stable" baseline (76%), so the proposal's ≥75% accuracy target is not met and is replaced by balanced accuracy
- **Limitation:** teaching dataset (anonymised); to be re-run on BioLINCC Framingham Offspring if access is granted